# Lesson 9 — Persistent RAG with sqlitesearch

Query notebook — reads from `faq.db` written by `sqlite-ingest.ipynb`.
No data fetching, no indexing. Just open the DB and query.

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
from sqlitesearch import TextSearchIndex
from rag_helper import RAGBase

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
# Connect to existing DB — no fit() needed
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

print(f"Documents in index: {sqlite_index.count()}")

In [ ]:
# Direct search — can run even while ingestion notebook is still running
results = sqlite_index.search(
    "Can I still join the course after it started?",
    num_results=5
)
[doc["question"] for doc in results]

In [ ]:
# Swap index into RAGBase — no other code changes needed
# This demonstrates the modular design: search backend is swappable
assistant = RAGBase(
    index=sqlite_index,
    llm_client=groq_client,
)

answer = assistant.rag("Can I still join the course after it started?")
print(answer)

In [ ]:
print(assistant.rag("How do I get a certificate?"))

In [ ]:
sqlite_index.close()